### Economy of negative climate impacts of drained peatland restoration

Samuli Launiainen & Anssi Ahtikoski Oct, 2025.

Scenario comparisons

1) Read RF calc outputs from files

2) Plot scenario comparisons for C dynamics

3) Plot scenario comparisons for RF's




In [81]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

# module main keeps track on C stocks and GHG fluxes between system components and the atmosphere
# calls functions from module radiative_forcing to compute RF's

#from main import wood_pools, soilCO2

# force iPython re-import modules at each call
%load_ext autoreload
%autoreload 2

EPS = 1e-6

#folder = r'Data/ES'
folder = r'Data/ES'

outname = r'Results/PS_Mtkg'
sitetype = 'Rhtkg'

sheets = ['FNR', 'FNR_WT_rise', 'Rest_RME', 'Rest_ROL', 'Rest_RSM']


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Run pipeline

In [82]:
# simulations in 'folder'
files = [name for name in os.listdir(folder) if '.xlsx' in name]
M = len(files)


aa = [name for name in files if stype in name]

scens = []
scennames = []
for sim in aa:
    f = os.path.join(folder, sim)
    
    for s in sheets[0:2]:
        tmp = pd.read_excel(f, sheet_name=s)
        scens.append(tmp)
        lab = s + ' ' + f.split('_')[-1].split('.')[0]
        scennames.append(lab)
        print(f, lab)

# restoration scenarios are same for all as initial state is same. So take them only from last sim
for s in sheets[2:]:
    tmp = pd.read_excel(f, sheet_name=s)
    scens.append(tmp)
    scennames.append(s)

res = dict(zip(scennames, scens))

# for c in ['C_tree', 'C_resid', 'C_WP_short', 'C_WP_long', 'C_soil', 'F_tree', 'F_resid', 'F_soil', 'F_WP_short', 'F_WP_long', 'F_CH4', 'F_N2O']:
#     plt.figure()
#     plt.plot(f0[c], 'r.-', label=ftype)
#     plt.plot(f1[c], 'g.-', label=rtype[0])
#     plt.plot(f2[c], 'b.-', label=rtype[1])
#     plt.plot(f3[c], 'k.-', label=rtype[2])   
#     plt.legend()
#     plt.title(c)

Data/ES\Ruotsinkylä_Rhtkg_CCF opt2%.xlsx FNR CCF opt2%
Data/ES\Ruotsinkylä_Rhtkg_CCF opt2%.xlsx FNR_WT_rise CCF opt2%
Data/ES\Ruotsinkylä_Rhtkg_CCF opt4%.xlsx FNR CCF opt4%
Data/ES\Ruotsinkylä_Rhtkg_CCF opt4%.xlsx FNR_WT_rise CCF opt4%
Data/ES\Ruotsinkylä_Rhtkg_RF opt2%.xlsx FNR RF opt2%
Data/ES\Ruotsinkylä_Rhtkg_RF opt2%.xlsx FNR_WT_rise RF opt2%
Data/ES\Ruotsinkylä_Rhtkg_RF opt4%.xlsx FNR RF opt4%
Data/ES\Ruotsinkylä_Rhtkg_RF opt4%.xlsx FNR_WT_rise RF opt4%
Data/ES\Ruotsinkylä_Rhtkg_suositusten mukaan.xlsx FNR suositusten mukaan
Data/ES\Ruotsinkylä_Rhtkg_suositusten mukaan.xlsx FNR_WT_rise suositusten mukaan


In [83]:
%matplotlib qt

import matplotlib.cm as cm
from matplotlib.colors import Normalize

# Define line styles separately
line_styles = {
    'FNR suositusten mukaan': '-',
    'FNR_WT_rise suositusten mukaan': '--',
    'FNR RF opt4%': '-', 
    'FNR_WT_rise RF opt4%': '--', 
    'FNR CCF opt4%': '-', 
    'FNR_WT_rise CCF opt4%': '--',
    'Rest_RME': ':', 
    'Rest_RSM': ':'
}

# Generate colors from colormap
scenario_keys = list(line_styles.keys())

# Create a mapping of base scenario names to colors
# Group scenarios by their base name (without WT_rise prefix)
base_scenarios = {}
for key in scenario_keys:
    base_key = key.replace('FNR_WT_rise ', 'FNR ')  # Normalize WT_rise variants to share colors
    if base_key not in base_scenarios:
        base_scenarios[base_key] = None

# Assign colors to base scenarios
n_base = len(base_scenarios)
cmap = cm.get_cmap('Set1')
norm = Normalize(vmin=0, vmax=n_base-1)

for i, base_key in enumerate(base_scenarios.keys()):
    base_scenarios[base_key] = cmap(norm(i))

# Build spec with shared colors for WT_rise variants
spec = {}
for key in scenario_keys:
    base_key = key.replace('FNR_WT_rise ', 'FNR ')
    color = base_scenarios[base_key]
    spec[key] = [color, line_styles[key]]

fig, ax = plt.subplots(2,1, figsize=(10,10))
fig2, ax2 = plt.subplots(2,2, figsize=(10,10))
fig3, ax3 = plt.subplots(2,1, figsize=(10,10))
fig4, ax4 = plt.subplots(2,1, figsize=(10,10))
fig5, ax5 = plt.subplots(1,2, figsize=(10,7))

for s in spec.keys():
    tmp = res[s]

    # at first timestep, total C is just the tree C
    tmp.loc[tmp.index[0],['C_resid', 'C_WP_short', 'C_WP_long']] = 0.0
    tmp['C_tot'] = tmp[['C_tree', 'C_resid', 'C_WP_short', 'C_WP_long', 'C_soil']].sum(axis=1) 
    tmp['C_tot'] -= tmp['C_tot'].loc[0]
    tmp['F_co2'] = tmp[['F_tree', 'F_resid', 'F_soil', 'F_WP_short', 'F_WP_long']].sum(axis=1) 
    tmp['RF_abo'] = tmp[['RF_tree', 'RF_resid', 'RF_WP_short', 'RF_WP_long']].sum(axis=1)
    tmp['C_abo'] = tmp[['C_tree', 'C_resid', 'C_WP_short', 'C_WP_long']].sum(axis=1)
    tmp['C_abo'] -= tmp['C_abo'].loc[0]
    print(s, tmp[['C_tot', 'C_tree', 'C_resid', 'C_WP_short', 'C_WP_long', 'C_soil']].mean(axis=0) )
    #ax[0,0].plot(tmp['F_co2'], '-', label=s); ax[0,0].set_title('Fco2')
    # if 'Rest' in s:
    #     ls = ':'
    # elif 'CCF' in s:
    #     ls = '--' 
    # else:
    #     ls = '-'
    col = spec[s][0]
    ls = spec[s][1]
    ax[0].plot(tmp['RF_tot'], linestyle = ls, color=col, label=s, alpha=0.9); 
    ax[0].set_ylabel(('$\Delta RF_{[0,t]}$ (W m$^{-2}$(earth) m$^{-2}$ (land))'))
    ax[1].plot(1e-3 * tmp['C_tot'], linestyle = ls, color=col, label=s, alpha=0.9); 
    ax[1].set_ylabel('$\Delta C_{tot}$ (kg C m$^{-2}$)')

    ax2[0,0].plot(1e-3 * tmp['C_tree'], linestyle = ls, color=col, label=s, alpha=0.9); ax2[0,0].set_title('$C_{tree}$ (kg C m$^{-2}$)')
    ax2[0,1].plot(1e-3 * tmp['C_resid'], linestyle = ls, color=col, label=s, alpha=0.9); ax2[0,1].set_title('$C_{resid}$ (kg C m$^{-2}$)')
    ax2[1,0].plot(1e-3 * tmp['C_soil'], linestyle = ls, color=col, label=s, alpha=0.9); ax2[1,0].set_title('$\Delta C_{soil}$ (kg C m$^{-2}$)')

    yy = 1e-3 * tmp[['C_WP_short', 'C_WP_long']].sum(axis=1)
    ax2[1,1].plot(yy, linestyle = ls, color=col, label=s, alpha=0.9); ax2[1,1].set_title('$C_{WP}$ (kg C m$^{-2}$)')
    del yy

    ax3[0].plot(tmp['RF_abo'], linestyle = ls, color=col, label=s, alpha=0.9);
    ax3[0].set_ylabel(('$\Delta RF_{[0,t]}$ (W m$^{-2}$(earth) m$^{-2}$ (land))'))
    ax3[0].set_title('tree + residues + WP only')
    ax[1].plot(1e-3 * tmp['C_tot'], linestyle = ls, color=col, label=s, alpha=0.9); 
    ax3[1].plot(1e-3* tmp['C_abo'], linestyle = ls, color=col, label=s, alpha=0.9); 
    ax3[1].set_title('C tree + residue + WP (kg C m$^{-2}$)')

    if ('Rest' not in s): # and ('WT_rise' not in s):

        mark = '.'
        col = spec[s][0]
        ls = spec[s][1]

        tmp = res[s]
        tt = np.arange(0, len(tmp))

        # at first timestep, total C is just the tree C
        tmp.loc[tmp.index[0],['C_resid', 'C_WP_short', 'C_WP_long']] = 0.0
        tmp['C_tot'] = tmp[['C_tree', 'C_resid', 'C_WP_short', 'C_WP_long', 'C_soil']].sum(axis=1) 

        tmp['F_co2'] = tmp[['F_tree', 'F_resid', 'F_soil', 'F_WP_short', 'F_WP_long']].sum(axis=1) 
        tmp['RF_abo'] = tmp[['RF_tree', 'RF_resid', 'RF_WP_short', 'RF_WP_long']].sum(axis=1)
        tmp['C_abo'] = tmp[['C_tree', 'C_resid', 'C_WP_short', 'C_WP_long']].sum(axis=1)
        tmp['productivity'] = -np.cumsum(tmp['F_tree'])

        #print(s, tmp[['C_tot', 'C_tree', 'C_resid', 'C_WP_short', 'C_WP_long', 'C_soil']].mean(axis=0) )


        y1 = np.cumsum(tmp['RF_tot']) / (tt + 1)
        y2 = np.cumsum(tmp['RF_abo']) / (tt + 1)
        x1 = -np.cumsum(tmp['F_tree'])

        ax4[0].plot(y1, linestyle = ls, color=col, label=s, alpha=0.9);
        ax4[0].set_ylabel('$\overline{\Delta RF}_{[0,t]}$ (W m$^{-2}$(earth) m$^{-2}$ (land))')
        ax4[1].plot(1e-3 * x1, linestyle = ls, color=col, label=s, alpha=0.9); 
        ax4[1].set_ylabel('Cumulative growth [kg C m$^{-2}$ (land)]')
        ax4[1].set_xlabel('time (yr)')
        ax4[1].legend(fontsize=8)

        ax5[0].plot(1e-3 * x1, y1, linestyle = ls, color=col, label=s, alpha=0.9);
        ax5[0].set_ylabel('$\overline{\Delta RF}_{[0,t]}$ (W m$^{-2}$(earth) m$^{-2}$ (land))')
        ax5[0].set_xlabel('Cumulative growth (kg C m$^{-2}$)')

        ax5[1].plot(1e-3 *x1, y2, linestyle = ls, color=col, label=s, alpha=0.9); 
        ax5[1].set_ylabel('$\overline{\Delta RF}_{[0,t]}$ (W m$^{-2}$(earth) m$^{-2}$ (land))')
        ax5[1].set_xlabel('Cumulative growth (kg C m$^{-2}$)')
        ax5[1].set_title('tree + residues + WP only')
        ax5[1].legend(fontsize=8)
    
ax[0].legend(fontsize=8)
ax2[1,0].legend(fontsize=8)
ax3[0].legend(fontsize=8)

fig.savefig(outname + '_RF.png')
fig2.savefig(outname + '_C.png')
fig3.savefig(outname + '_RF_ag.png')
fig4.savefig(outname + '_RFpast.png')
fig5.savefig(outname + '_RF_vs_productivity.png')

C:\Users\03081268\AppData\Local\Temp\ipykernel_21920\4196913142.py:31: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap('Set1')


FNR suositusten mukaan C_tot        -19310.008706
C_tree         6929.718406
C_resid        1224.379078
C_WP_short      489.702825
C_WP_long      1846.303355
C_soil       -16022.325893
dtype: float64
FNR_WT_rise suositusten mukaan C_tot        -11275.436488
C_tree         6929.718406
C_resid        1224.379078
C_WP_short      489.702825
C_WP_long      1846.303355
C_soil        -7920.451673
dtype: float64
FNR RF opt4% C_tot        -18611.419065
C_tree         5816.305667
C_resid        1700.237301
C_WP_short      690.956077
C_WP_long      1769.042579
C_soil       -14810.159522
dtype: float64
FNR_WT_rise RF opt4% C_tot        -11169.718413
C_tree         5816.305667
C_resid        1700.237301
C_WP_short      690.956077
C_WP_long      1769.042579
C_soil        -7301.156865
dtype: float64
FNR CCF opt4% C_tot        -23698.322992
C_tree         5531.391266
C_resid        1035.825919
C_WP_short      465.489332
C_WP_long      1328.617366
C_soil       -18281.845708
dtype: float64
FNR_WT_rise C

In [68]:
130/460.


0.2826086956521739

In [13]:
scennames

['FNR CCF opt2%',
 'FNR_WT_rise CCF opt2%',
 'FNR CCF opt4%',
 'FNR_WT_rise CCF opt4%',
 'FNR RF opt2%',
 'FNR_WT_rise RF opt2%',
 'FNR RF opt4%',
 'FNR_WT_rise RF opt4%',
 'FNR suositusten mukaan',
 'FNR_WT_rise suositusten mukaan',
 'Rest_RME',
 'Rest_ROL',
 'Rest_RSM']

In [35]:
fig4, ax4 = plt.subplots(2,1, figsize=(10,10))
fig4, ax5 = plt.subplots(1,2, figsize=(10,7))


for s in spec.keys():

    if ('Rest' not in s): # and ('WT_rise' not in s):

        mark = '.'
        col = spec[s][0]
        ls = spec[s][1]

        tmp = res[s]
        tt = np.arange(0, len(tmp))

        # at first timestep, total C is just the tree C
        tmp.loc[tmp.index[0],['C_resid', 'C_WP_short', 'C_WP_long']] = 0.0
        tmp['C_tot'] = tmp[['C_tree', 'C_resid', 'C_WP_short', 'C_WP_long', 'C_soil']].sum(axis=1) 

        tmp['F_co2'] = tmp[['F_tree', 'F_resid', 'F_soil', 'F_WP_short', 'F_WP_long']].sum(axis=1) 
        tmp['RF_abo'] = tmp[['RF_tree', 'RF_resid', 'RF_WP_short', 'RF_WP_long']].sum(axis=1)
        tmp['C_abo'] = tmp[['C_tree', 'C_resid', 'C_WP_short', 'C_WP_long']].sum(axis=1)
        tmp['productivity'] = -np.cumsum(tmp['F_tree'])

        #print(s, tmp[['C_tot', 'C_tree', 'C_resid', 'C_WP_short', 'C_WP_long', 'C_soil']].mean(axis=0) )


        y1 = np.cumsum(tmp['RF_tot']) / (tt + 1)
        y2 = np.cumsum(tmp['RF_abo']) / (tt + 1)
        x1 = -np.cumsum(tmp['F_tree'])

        ax4[0].plot(y1, linestyle = ls, color=col, label=s, alpha=0.6);
        ax4[0].set_ylabel('$\overline{\Delta RF}_{[0,t]}$ (W m$^{-2}$(earth) m$^{-2}$ (land))')
        ax4[1].plot(1e-3 * x1, linestyle = ls, color=col, label=s, alpha=0.6); 
        ax4[1].set_ylabel('Cumulative growth [kg C m$^{-2}$ (land)]')
        ax4[1].set_xlabel('time (yr)')
        ax4[1].legend()

        ax5[0].plot(1e-3 * x1, y1, linestyle = ls, color=col, label=s, alpha=0.6);
        ax5[0].set_ylabel('$\overline{\Delta RF}_{[0,t]}$ (W m$^{-2}$(earth) m$^{-2}$ (land))')
        ax5[0].set_xlabel('Cumulative growth (kg C m$^{-2}$)')

        ax5[1].plot(1e-3 *x1, y2, linestyle = ls, color=col, label=s, alpha=0.6); 
        ax5[1].set_ylabel('$\overline{\Delta RF_{tree + resid + WP}}_{[0,t]}$ (W m$^{-2}$(earth) m$^{-2}$ (land))')
        ax5[1].set_xlabel('Cumulative growth (kg C m$^{-2}$)')
        ax5[1].legend()
        # ax5[0].plot(1e-3 * tmp['productivity'], y1, marker=mark, color=col, label=s, alpha=0.6); ax5[0].set_title('past mean RF tot')
        # ax5[1].plot(1e-3 *tmp['productivity'], y2, marker=mark, color=col, label=s, alpha=0.6); 
        #ax5[1].set_title('RF tree + residue + WP (CO2 only)')